01 - Build the PNG mosaic corpus
#
# Converts the mounted DICOM series into one lossless PNG mosaic per selected
# series (~8-9 GB for the full training set, inside the 20 GB output cap).
# Save a Version when this finishes; the output becomes the dataset that the
# training notebook reads.
#
# It is resumable: if the 12 h session limit hits, re-run and it skips what is
# already written.

In [ ]:
import sys, subprocess
sys.path.insert(0, "/kaggle/input/rsna-knee-code/src")
sys.path.insert(0, "/kaggle/input/rsna-knee-code")
# timm is not always present in the Kaggle image; install from the offline wheel
# dataset when the competition notebook has no internet.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm
import rsna_knee.config as cfg
print("ON_KAGGLE", cfg.ON_KAGGLE)
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP)

In [ ]:
!cd /kaggle/input/rsna-knee-code && python scripts/build_weak_labels.py

In [ ]:
# Full corpus. Raise --workers to the number of available cores.
!cd /kaggle/input/rsna-knee-code && python scripts/preprocess_series.py --split train --workers 4

In [ ]:
!cd /kaggle/input/rsna-knee-code && python scripts/preprocess_series.py --split test --workers 4

In [ ]:
from pathlib import Path
prep = Path("/kaggle/working/prep")
pngs = list(prep.rglob("*.png"))
size = sum(p.stat().st_size for p in pngs)
print(f"{len(pngs)} mosaics, {size/1e9:.2f} GB")